# 04 · Analisis exploratorio

> **Objetivo:** entender la distribucion de las variables, la temporalidad
> del negocio y las asociaciones entre `Quantity`, `Price` y los agregados
> por cliente. Reproducimos las Figuras 6–9 y las Tablas 4, 5 y 7 del OVA.

## Que vamos a hacer

1. Univariado: cuartiles y cercas de Tukey para `Price` y `Quantity`.
2. Temporal: ventas por mes, mapa dia × hora.
3. Asociaciones: Pearson y Spearman entre `Quantity` y `Price`.
4. Nivel cliente: Spearman frecuencia–monetario y participacion del 20 %
   de mayor valor.

> Si quieres las figuras finales con la paleta del OVA, usa
> `scripts/figuras_u2.py`; aqui nos quedamos con tablas para discutir.

In [ ]:
# Permite importar el paquete segmentacion desde el notebook
import sys
from pathlib import Path

_cwd = Path.cwd()
if _cwd.name == "legacy":
    PROJECT_ROOT = _cwd.parent.parent
elif _cwd.name == "notebooks":
    PROJECT_ROOT = _cwd.parent
else:
    PROJECT_ROOT = _cwd
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

In [ ]:
import polars as pl

from segmentacion.config import PROCESSED_DATA_DIR
from segmentacion.data.cargar import leer_parquet
from segmentacion.eda.resumen import (
    asociaciones,
    cercas_tukey,
    cuartiles,
    matriz_dia_hora,
    ventas_por_mes,
)

PARQUET_VENTAS = PROCESSED_DATA_DIR / "ventas_validas.parquet"
assert PARQUET_VENTAS.exists(), "Ejecuta `02_limpieza.ipynb` primero."
ventas = leer_parquet(PARQUET_VENTAS)
print(f"ventas validas: {ventas.height:,}".replace(",", " "))

## 1. Univariado — `Price` y `Quantity`

In [ ]:
print("Price:", cuartiles(ventas, "Price"))
print("Quantity:", cuartiles(ventas, "Quantity"))

### Cercas de Tukey

El OVA cita: `Price` → cerca superior 7,50 (8,40 % la superan); `Quantity`
→ cerca superior 27 (6,56 %). Son los umbrales que usaremos en U3 para
decidir si "cortar" o "transformar" los outliers.

In [ ]:
inf_p, sup_p, frac_p = cercas_tukey(ventas, "Price")
inf_q, sup_q, frac_q = cercas_tukey(ventas, "Quantity")
print(f"Price   -> cerca sup = {sup_p:.2f}; fraccion > sup = {frac_p:.2%}")
print(f"Quantity-> cerca sup = {sup_q:.2f}; fraccion > sup = {frac_q:.2%}")

## 2. Temporal — ventas por mes y mapa dia × hora

In [ ]:
mes = ventas_por_mes(ventas)
mes.tail(6)

In [ ]:
hm = matriz_dia_hora(ventas)
# Pivot a matriz 7 x H para inspeccion visual rapida.
hm.pivot(index="dia", on="hora", values="n", aggregate_function="sum").sort("dia")

Confirmamos el patron del OVA: pico en **septiembre-noviembre** y
practicamente nada el **sabado** (400 filas en dos años).

In [ ]:
sab = ventas.filter(pl.col("InvoiceDate").dt.weekday() == 6).height
print(f"filas en sabado: {sab}")

## 3. Asociaciones — Quantity frente a Price

In [ ]:
asoc = asociaciones(ventas, "Quantity", "Price")
print(f"Pearson  : {asoc['pearson']:.3f}")
print(f"Spearman : {asoc['spearman']:.3f}")

La **Spearman** fuertemente negativa (−0,428) frente a una
**Pearson** casi nula (−0,005) es el ejemplo del OVA: el orden importa,
no el valor absoluto. En productos de regalo caros se vende poca unidad;
en los baratos muchas.

## 4. Nivel cliente — frecuencia vs monetario

In [ ]:
por_cliente = ventas.group_by("Customer ID").agg(
    pl.col("Invoice").n_unique().alias("frecuencia"),
    pl.col("Importe").sum().alias("monetario"),
)
rho = por_cliente.select(pl.corr("frecuencia", "monetario", method="spearman")).item()
print(f"Spearman frecuencia-monetario: {rho:.3f}")

In [ ]:
por_cliente_ord = por_cliente.sort("monetario", descending=True)
n_top = int(0.2 * por_cliente_ord.height)
share = por_cliente_ord.head(n_top)['monetario'].sum() / por_cliente_ord['monetario'].sum()
print(f"participacion del 20% mayor: {share:.2%}")

## Hallazgos para `HALLAZGOS.md`

| Hallazgo | Evidencia | Decision |
|---|---|---|
| Precio con cola larga | Cerca de Tukey = 7,50; 8,4 % la supera | Log1p en U3 |
| Cantidad con cola larga | Cerca de Tukey = 27; 6,56 % la supera | Log1p en U3 |
| Pearson ~ 0 pero Spearman −0,43 | Pearson −0,005; Spearman −0,428 | Priorizar ranks (features) |
| Concentracion fuerte en pocos clientes | 20 % superior = 77,2 % | Segmentar antes de ofertar |
| Sabado casi inexistente | 400 filas en dos años | Ignorar al modelar estacionalidad semanal |

> **Siguiente paso:** `03_variables_cliente.ipynb` — construimos RFM
> con fecha de corte, el **puente** a la Unidad 3.